# Algoritmos da Disciplina: congress-Twitter (G24)



## Verificar o ambiente

In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\lucas\OneDrive\Área de Trabalho\analise-congress-twitter-g24\.venv\Scripts\python.exe
3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]


## Carregar o grafo

In [2]:
import math
import networkx as nx

caminho = "dados/congress_network/congress.edgelist"
G = nx.read_edgelist(caminho, create_using=nx.DiGraph, nodetype=int)  # grafo dirigido
n = G.number_of_nodes()

# Custo de cada aresta: -log(probabilidade). Maior influência = menor custo.
for u, v, d in G.edges(data=True):
    d["custo"] = -math.log(d["weight"])

# Versão não dirigida: A->B e B->A viram uma aresta só, com a maior das duas probabilidades.
G_nd = nx.Graph()
G_nd.add_nodes_from(G.nodes())
for u, v, d in G.edges(data=True):
    p = d["weight"]
    if G_nd.has_edge(u, v):
        p = max(p, G_nd[u][v]["weight"])
    G_nd.add_edge(u, v, weight=p, custo=-math.log(p))

print(f"Vértices: {n}, arestas (dirigido): {G.number_of_edges()}, arestas (não dirigido): {G_nd.number_of_edges()}")

Vértices: 475, arestas (dirigido): 13289, arestas (não dirigido): 10222


## Medição de tempo

In [3]:
import time
import statistics
from scipy import stats

tempos = {}

def medir_tempo(funcao, repeticoes=30):
    lista = []
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        funcao()
        lista.append(time.perf_counter() - inicio)
    return lista

def mostrar_tempos(lista, colunas=10):
    print("Tempos (ms):")
    for i in range(0, len(lista), colunas):
        print("  ".join(f"{t * 1000:10.4f}" for t in lista[i:i + colunas]))

def resumir(lista):
    k = len(lista)
    media = statistics.mean(lista)
    desvio = statistics.stdev(lista)
    critico = 1.96 if k >= 30 else stats.t.ppf(0.975, k - 1)  # normal (k >= 30) ou t-Student
    margem = critico * desvio / k ** 0.5
    print(f"Média: {media * 1000:.4f} ms, desvio padrão: {desvio * 1000:.4f} ms")
    print(f"IC 95%: [{(media - margem) * 1000:.4f}; {(media + margem) * 1000:.4f}] ms")

def cronometrar(nome, funcao):
    tempos[nome] = medir_tempo(funcao)
    mostrar_tempos(tempos[nome])
    resumir(tempos[nome])

print("Funções de medição prontas.")

Funções de medição prontas.


## Busca em largura (BFS)

In [4]:
arestas = list(nx.bfs_edges(G, 0))
print(f"Alcança {len(arestas) + 1} vértices a partir do vértice 0.")

cronometrar("BFS", lambda: list(nx.bfs_edges(G, 0)))

Alcança 469 vértices a partir do vértice 0.
Tempos (ms):
    2.8177      2.5162      2.4855      2.1644      2.1925      1.8514      1.5394      1.5886      1.6440      2.1983
    1.8992      1.8315      1.6147      1.6289      1.6901      1.6418      1.6934      1.8146      2.2932      1.8940
    1.4502      1.3811      1.3814      1.3312      1.3606      1.3512      1.3193      1.3640      1.5915      2.7212
Média: 1.8084 ms, desvio padrão: 0.4306 ms
IC 95%: [1.6543; 1.9625] ms


## Busca em profundidade (DFS)

In [5]:
arestas = list(nx.dfs_edges(G, 0))
print(f"Alcança {len(arestas) + 1} vértices a partir do vértice 0.")

cronometrar("DFS", lambda: list(nx.dfs_edges(G, 0)))

Alcança 469 vértices a partir do vértice 0.
Tempos (ms):
    3.3093      2.8318      2.6364      2.5414      2.6843      3.8119      3.2796      2.6142      2.4719      2.4501
    2.5321      2.4014      2.2648      2.7675      2.3507      2.2189      2.2410      2.3033      2.4618      3.6111
    2.7339      2.2638      8.1115      2.7262      2.7248      2.5254      2.4425      3.0988      2.6439      5.6784
Média: 2.9578 ms, desvio padrão: 1.1839 ms
IC 95%: [2.5341; 3.3814] ms


## Verificação de eulerianidade

In [6]:
print(f"Dirigido: {'euleriano' if nx.is_eulerian(G) else 'não euleriano'}")
print(f"Não dirigido: {'euleriano' if nx.is_eulerian(G_nd) else 'não euleriano'}")

cronometrar("Eulerianidade (orientado)", lambda: nx.is_eulerian(G))
cronometrar("Eulerianidade (não orientado)", lambda: nx.is_eulerian(G_nd))

Dirigido: não euleriano
Não dirigido: não euleriano
Tempos (ms):
    0.0339      0.0141      0.0084      0.0083      0.0078      0.0096      0.0105      0.0093      0.0070      0.0069
    0.0068      0.0068      0.0058      0.0060      0.0072      0.0063      0.0068      0.0080      0.0073      0.0072
    0.0082      0.0061      0.0064      0.0065      0.0061      0.0062      0.0074      0.0059      0.0056      0.0078
Média: 0.0083 ms, desvio padrão: 0.0051 ms
IC 95%: [0.0065; 0.0102] ms
Tempos (ms):
    0.0306      0.0089      0.0080      0.0091      0.0089      0.0087      0.0071      0.0078      0.0067      0.0073
    0.0057      0.0061      0.0060      0.0060      0.0058      0.0055      0.0058      0.0065      0.0059      0.0060
    0.0060      0.0062      0.0056      0.0091      0.0081      0.0098      0.0081      0.0108      0.0063      0.0066
Média: 0.0080 ms, desvio padrão: 0.0045 ms
IC 95%: [0.0064; 0.0096] ms


## Dijkstra

In [7]:
distancias = nx.single_source_dijkstra_path_length(G, 0, weight="custo")
print(f"Alcança {len(distancias)} vértices a partir do vértice 0.")
print(f"Maior custo: {max(distancias.values()):.4f}")

cronometrar("Dijkstra", lambda: nx.single_source_dijkstra_path_length(G, 0, weight="custo"))

Alcança 469 vértices a partir do vértice 0.
Maior custo: 21.9834
Tempos (ms):
   21.2722     11.9731     48.9982      9.9426     10.0830      7.7600      8.7753      7.2271      9.0718      6.7073
    8.1425      6.7987      9.9329     10.1729      7.4516      6.5309      7.9875      6.4842      7.5299      6.6009
    6.9714      8.8467      8.1995      7.8419      6.2543      7.8804      6.2046      8.3308      8.1666      6.0883
Média: 9.8076 ms, desvio padrão: 7.9127 ms
IC 95%: [6.9760; 12.6391] ms


## Bellman-Ford

In [8]:
distancias = nx.single_source_bellman_ford_path_length(G, 0, weight="custo")
print(f"Alcança {len(distancias)} vértices a partir do vértice 0.")
print(f"Maior custo: {max(distancias.values()):.4f}")

cronometrar("Bellman-Ford", lambda: nx.single_source_bellman_ford_path_length(G, 0, weight="custo"))

Alcança 469 vértices a partir do vértice 0.
Maior custo: 21.9834
Tempos (ms):
   10.1369     12.6900     13.4741      9.5212      8.8812      8.8861      7.6595     11.0754      7.6456     10.4037
   10.1873      9.7766      9.8528     10.2182     11.3940      8.5871     12.1585     11.4122      7.7062     10.6185
    9.2759      8.4644     10.1810      9.2796      8.1908      6.9948      7.9787      7.4556     10.0037      8.5479
Média: 9.6219 ms, desvio padrão: 1.6071 ms
IC 95%: [9.0468; 10.1970] ms


## Floyd-Warshall

In [9]:
import numpy as np

distancias = nx.floyd_warshall_numpy(G, nodelist=range(n), weight="custo")
print(f"Matriz de distâncias: {distancias.shape[0]} x {distancias.shape[1]}")
print(f"Maior custo: {distancias[np.isfinite(distancias)].max():.4f}")

cronometrar("Floyd-Warshall", lambda: nx.floyd_warshall_numpy(G, nodelist=range(n), weight="custo"))

Matriz de distâncias: 475 x 475
Maior custo: 32.1171
Tempos (ms):
 1285.3502   1290.3461   1211.1641   1319.7632   1182.8674   1506.4441   1232.5131   1301.5379   1105.8631   1210.9980
 1661.1272   1149.6828   1184.7206   1147.2454   1106.6959   1094.4893   1029.5252   1010.0881    977.9258    959.0013
 1197.3677   1054.6919   1063.3311   1056.9412   1012.9376   1087.8715   1057.1235   1457.2705   1267.1875   1229.4319
Média: 1181.7168 ms, desvio padrão: 160.2585 ms
IC 95%: [1124.3690; 1239.0646] ms


## Algoritmo de Tarjan

In [10]:
componentes = sorted(nx.strongly_connected_components(G), key=len, reverse=True)
print(f"Componentes fortemente conexas: {len(componentes)}")
print(f"Tamanhos: {', '.join(str(len(x)) for x in componentes)}")

cronometrar("Tarjan", lambda: list(nx.strongly_connected_components(G)))

Componentes fortemente conexas: 7
Tamanhos: 469, 1, 1, 1, 1, 1, 1
Tempos (ms):
   10.0071      9.3541      8.1774      8.6474      8.1015     12.3398     10.8518      8.3691      9.3621      7.7651
    7.5917      7.3197      8.8436      9.4986     11.1958     12.9037      8.7307     11.1147      7.7347     11.1419
   12.2308      8.3214     10.4999     11.4450      7.5315      8.4089      9.2774      9.3993      7.5778      9.3265
Média: 9.4356 ms, desvio padrão: 1.5927 ms
IC 95%: [8.8657; 10.0056] ms


## Árvore geradora mínima (Kruskal)

In [11]:
arvore = nx.minimum_spanning_tree(G_nd, weight="custo", algorithm="kruskal")
print(f"Arestas na árvore: {arvore.number_of_edges()}")
print(f"Custo total: {arvore.size(weight='custo'):.4f}")

cronometrar("Kruskal", lambda: nx.minimum_spanning_tree(G_nd, weight="custo", algorithm="kruskal"))

Arestas na árvore: 474
Custo total: 1748.5905
Tempos (ms):
   72.0781     39.1516    111.0156     35.2414     44.0329     39.0775     35.3868     33.3471     40.2422     31.9390
   34.1841     38.9870    114.1878     53.8771     54.2568     45.1833     40.7039     37.7937     36.9970     35.7756
   35.7604     34.7891    124.9128     42.8617     36.8508     38.5362     37.1426     43.5268     43.5753     48.5765
Média: 48.6664 ms, desvio padrão: 24.4711 ms
IC 95%: [39.9095; 57.4232] ms


## Árvore geradora mínima (Prim)

In [12]:
arvore = nx.minimum_spanning_tree(G_nd, weight="custo", algorithm="prim")
print(f"Arestas na árvore: {arvore.number_of_edges()}")
print(f"Custo total: {arvore.size(weight='custo'):.4f}")

cronometrar("Prim", lambda: nx.minimum_spanning_tree(G_nd, weight="custo", algorithm="prim"))

Arestas na árvore: 474
Custo total: 1748.5905
Tempos (ms):
   31.6835     29.5982     89.3940     28.3282     39.9411     49.3636     54.6531     53.7975     65.3168     47.9345
   36.3424     53.6586     55.4782     86.7039     63.1533    129.3394     27.2303     32.9987     25.5815     19.7507
   30.0558     39.8049     37.9751     37.2296     45.3431     32.4420    123.4195     37.6735     45.9141     48.6178
Média: 49.9574 ms, desvio padrão: 26.4012 ms
IC 95%: [40.5099; 59.4050] ms


## Exportar tempos

In [13]:
import csv
import os

if not tempos:
    print("Nenhum tempo medido ainda: rode as seções dos algoritmos antes de exportar.")
else:
    os.makedirs("resultados", exist_ok=True)
    arquivo = "resultados/tempos_algoritmos.csv"
    with open(arquivo, "w", newline="", encoding="utf-8") as f:
        escritor = csv.writer(f)
        escritor.writerow(["algoritmo", "execucao", "tempo_ms"])
        for algoritmo, lista in tempos.items():
            for i, t in enumerate(lista, start=1):
                escritor.writerow([algoritmo, i, f"{t * 1000:.4f}"])
    with open("resultados/stats_algoritmos.csv", "w", newline="", encoding="utf-8") as f:
        escritor = csv.writer(f)
        escritor.writerow(["algoritmo", "n", "media_ms", "desvio_ms", "ic95_inf_ms", "ic95_sup_ms"])
        for algoritmo, lista in tempos.items():
            k = len(lista)
            media = statistics.mean(lista) * 1000
            desvio = statistics.stdev(lista) * 1000
            critico = 1.96 if k >= 30 else stats.t.ppf(0.975, k - 1)
            margem = critico * desvio / k ** 0.5
            escritor.writerow([algoritmo, k, f"{media:.4f}", f"{desvio:.4f}", f"{media - margem:.4f}", f"{media + margem:.4f}"])

    print(f"Algoritmos exportados: {', '.join(tempos)}")
    print(f"{sum(len(l) for l in tempos.values())} tempos salvos em {arquivo}")
    print("Resumo (média, desvio padrão e IC 95%) salvo em resultados/stats_algoritmos.csv")

Algoritmos exportados: BFS, DFS, Eulerianidade (orientado), Eulerianidade (não orientado), Dijkstra, Bellman-Ford, Floyd-Warshall, Tarjan, Kruskal, Prim
300 tempos salvos em resultados/tempos_algoritmos.csv
Resumo (média, desvio padrão e IC 95%) salvo em resultados/stats_algoritmos.csv
